# Experiment H25a: LayerNorm, balance ratio, and Muon on a toy nonlinear regression

This notebook is the narrative counterpart of `run_experiment.py` in the same directory. It **imports and uses the script's experiment runner directly** so that the notebook and script stay computationally aligned.

## Pair-level identity
- **Script counterpart:** `experiments/Tier_2_Symmetry_Reparametrization_Tests/H25a_LAYERNORM_SAFE_ZONE/run_experiment.py`
- **Question studied here:** in this **single-seed synthetic nonlinear regression** setup, how do SGD vs Muon behave with and without LayerNorm when judged by loss and the balance ratio
  \[
  c(t) = \frac{\max_\ell \lVert W_\ell \rVert_F}{\min_\ell \lVert W_\ell \rVert_F}?
  \]
- **Toy-scope framing:** this is a mechanistic probe, **not** a transformer benchmark and not statistical evidence across seeds.

The first completion pass below keeps the original experimental design as much as possible while fixing several rigor issues from the earlier version:
1. the script is import-safe and reusable;
2. all four conditions use the **same shared base initialization** by default;
3. `c(t)` is tracked at **every optimization step** rather than only every 50 steps;
4. the notebook presents only quantities that are actually computed by the script.

## Measured vs not measured

### Measured in this notebook/script pair
- full-batch training loss over time;
- balance ratio `c(t)` over time;
- final per-layer Frobenius norms;
- divergence flags;
- four summary tests `T1`–`T4` using the raw metrics returned by the script.

### Not measured here
- transformer task performance;
- uncertainty across random seeds;
- any formal gauge quantity;
- gradient-norm or activation-statistics mechanisms;
- any causal claim beyond the recorded toy metrics.

Interpretation should therefore stay narrow: this notebook can support statements about the observed toy run, but not broad claims about transformer-scale success.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time
import textwrap

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})
np.set_printoptions(precision=4, suppress=True)

SCRIPT_RELATIVE = Path("experiments/Tier_2_Symmetry_Reparametrization_Tests/H25a_LAYERNORM_SAFE_ZONE/run_experiment.py")
NOTEBOOK_LOCAL = Path("run_experiment.py")

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
candidates = []
for root in search_roots:
    candidates.append(root / SCRIPT_RELATIVE)
    candidates.append(root / NOTEBOOK_LOCAL)

SCRIPT_PATH = None
for candidate in candidates:
    if candidate.exists():
        SCRIPT_PATH = candidate.resolve()
        break

if SCRIPT_PATH is None:
    raise FileNotFoundError(
        "Could not locate run_experiment.py from the notebook. Checked: "
        + ", ".join(str(c) for c in candidates)
    )

NOTEBOOK_DIR = SCRIPT_PATH.parent
REPO_ROOT = next(
    (parent for parent in [SCRIPT_PATH.parent, *SCRIPT_PATH.parents] if (parent / "experiments").exists()),
    SCRIPT_PATH.parent,
)

spec = importlib.util.spec_from_file_location("h25a_run_experiment", SCRIPT_PATH)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

print("Notebook/script path resolution")
print("-" * 32)
print(f"Working directory : {Path.cwd().resolve()}")
print(f"Repo root         : {REPO_ROOT}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Script path       : {SCRIPT_PATH}")

## Reproducibility, runtime, and exact default configuration

The next cell runs the experiment through the imported script API. This is the main alignment guarantee for the pair: all numerical results shown below come from the same code path that the script uses when executed directly.

In [ ]:
notebook_start = time.time()
experiment = module.run_all()
notebook_elapsed = time.time() - notebook_start

meta = experiment["meta"]
results = experiment["results"]
tests = experiment["tests"]
data_stats = experiment["data"]

assert meta["shared_init"] is True
assert meta["shared_init_verified"] is True, "Shared initialization should be verified across all four configs."
assert len(results["Muon+LN"]["c"]) == meta["dimensions"]["n_steps"] + 1
assert len(results["Muon+LN"]["loss"]) == meta["dimensions"]["n_steps"] + 1

print("Reproducibility / runtime / configuration")
print("-" * 44)
print(f"Seed                     : {meta['seed']}")
print(f"Shared initialization    : {meta['shared_init']} (verified={meta['shared_init_verified']})")
print(f"Config order             : {meta['config_order']}")
print(f"Dimensions               : {meta['dimensions']}")
print(f"Display report steps     : {meta['report_steps']}")
print(f"Display sampling interval: every {meta['display_every']} steps")
print(f"Safe-zone threshold      : c < {meta['safe_zone_c']}")
print(f"Dataset shapes           : X={data_stats['x_shape']}, Y={data_stats['y_shape']}")
print(f"Input norm mean ± std    : {data_stats['input_norm_mean']:.4f} ± {data_stats['input_norm_std']:.4f}")
print(f"Target Frobenius norm    : {data_stats['target_fro']:.4f}")
print(f"Script runtime           : {meta['runtime_sec']:.3f}s")
print(f"Notebook run_all runtime : {notebook_elapsed:.3f}s")
print(f"Python                   : {platform.python_version()}")
print(f"NumPy                    : {np.__version__}")
print(f"Matplotlib               : {plt.matplotlib.__version__}")

## Exact experimental setup actually executed

The imported script preserves the original toy setup:
- one synthetic dataset with `100` samples and feature width `32`;
- a `4`-block network with `8` weight matrices total;
- full-batch training for `500` steps at learning rate `0.005`;
- four conditions: `SGD+LN`, `SGD-LN`, `Muon+LN`, `Muon-LN`;
- Muon uses the exact SVD polar factor of each weight gradient;
- LayerNorm parameters, when present, are updated with plain gradient descent.

What changed for rigor in this pass is **not** the scientific question but the bookkeeping and truthfulness of the comparison.

In [ ]:
def fmt(value, digits=4):
    if isinstance(value, bool):
        return str(value)
    if value is None:
        return "None"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if np.isinf(value):
        return "INF"
    if np.isnan(value):
        return "nan"
    return f"{float(value):.{digits}f}"


def print_table(headers, rows):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    line = " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(line)
    print(sep)
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


summary_headers = ["config", "optimizer", "layernorm", "final_loss", "max_c", "final_c", "diverged"]
summary_rows = []
for name in meta["config_order"]:
    run = results[name]
    summary_rows.append([
        name,
        "Muon" if run["use_muon"] else "SGD",
        "yes" if run["use_ln"] else "no",
        fmt(run["final_loss"]),
        fmt(run["max_c"]),
        fmt(run["final_c"]),
        str(run["diverged"]),
    ])

raw_test_headers = ["test", "pass", "raw values"]
raw_test_rows = [
    [
        "T1",
        str(tests["T1"]["pass"]),
        f"max c(Muon+LN)={fmt(tests['T1']['raw']['muon_ln_max_c'])}, final c(Muon+LN)={fmt(tests['T1']['raw']['muon_ln_final_c'])}",
    ],
    [
        "T2",
        str(tests["T2"]["pass"]),
        (
            f"SGD max-c ratio noLN/+LN={fmt(tests['T2']['raw']['sgd_max_c_ratio_no_ln_over_plus_ln'], 2)}x, "
            f"Muon max-c ratio noLN/+LN={fmt(tests['T2']['raw']['muon_max_c_ratio_no_ln_over_plus_ln'], 2)}x, "
            f"SGD-LN diverged={tests['T2']['raw']['sgd_noln_diverged']}"
        ),
    ],
    [
        "T3",
        str(tests["T3"]["pass"]),
        (
            f"loss(SGD+LN)={fmt(tests['T3']['raw']['sgd_plus_ln_final_loss'])}, "
            f"loss(Muon+LN)={fmt(tests['T3']['raw']['muon_plus_ln_final_loss'])}, "
            f"advantage ratio SGD/Muon={fmt(tests['T3']['raw']['advantage_ratio_sgd_over_muon'], 2)}x"
        ),
    ],
    [
        "T4",
        str(tests["T4"]["pass"]),
        (
            f"SGD loss ratio noLN/+LN={fmt(tests['T4']['raw']['sgd_loss_ratio_no_ln_over_plus_ln'], 2)}x, "
            f"Muon loss ratio noLN/+LN={fmt(tests['T4']['raw']['muon_loss_ratio_no_ln_over_plus_ln'], 2)}x, "
            f"asymmetry ratio={fmt(tests['T4']['raw']['asymmetry_ratio_sgd_penalty_over_muon_penalty'], 2)}x"
        ),
    ],
]

print("Compact configuration summary")
print("=" * 80)
print_table(summary_headers, summary_rows)
print()
print("Raw T1-T4 metrics")
print("=" * 80)
print_table(raw_test_headers, raw_test_rows)
print()

ranking = sorted((results[name]["final_loss"], name) for name in meta["config_order"])
ranking_text = " < ".join(f"{name} ({fmt(loss)})" for loss, name in ranking)
print(f"Final-loss ranking (lower is better): {ranking_text}")
print(f"Overall test score: {tests['summary']['n_pass']}/{tests['summary']['score_out_of']}")

In [ ]:
muon_ln_loss = results["Muon+LN"]["final_loss"]
muon_noln_loss = results["Muon-LN"]["final_loss"]
muon_ln_c = results["Muon+LN"]["max_c"]
muon_noln_c = results["Muon-LN"]["max_c"]

print("Immediate interpretation of the compact tables")
print("-" * 48)
print(f"Muon+LN max c : {muon_ln_c:.4f}")
print(f"Muon-LN max c : {muon_noln_c:.4f}")
print(f"Muon+LN loss  : {muon_ln_loss:.4f}")
print(f"Muon-LN loss  : {muon_noln_loss:.4f}")
print()
if muon_noln_loss < muon_ln_loss:
    print("In this verified seed-42 run, Muon-LN reaches a lower final loss than Muon+LN.")
    print("Therefore the notebook should not claim that LN+Muon is the best-performing combination on final loss.")
else:
    print("In this verified seed-42 run, Muon+LN reaches the lowest Muon final loss.")
print("The narrower supported statement is that Muon+LN shows a lower observed c(t) trajectory than Muon-LN in this toy setup.")

## Balance-ratio trajectories

The next figure shows the balance ratio `c(t)` for all four configurations, with the historical safe-zone reference line at `c = 2`.

Because the script now stores `c(t)` at **every step**, the visual and the corresponding `T1` statement are honest with respect to the implemented metric.

In [ ]:
plt.figure(figsize=(9, 4.8))
for name in meta["config_order"]:
    steps = np.array(results[name]["step"])
    c_vals = np.array(results[name]["c"])
    finite = np.isfinite(c_vals)
    plt.plot(steps[finite], c_vals[finite], label=name, linewidth=2)
plt.axhline(meta["safe_zone_c"], color="black", linestyle="--", linewidth=1.5, label="safe-zone c=2")
plt.xlabel("training step")
plt.ylabel("balance ratio c(t)")
plt.title("H25a balance-ratio trajectories")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Interpretation of c(t)")
print("-" * 24)
print(f"T1 pass: {tests['T1']['pass']}")
print(f"Muon+LN max c over every recorded step: {tests['T1']['raw']['muon_ln_max_c']:.4f}")
print(f"Muon-LN / Muon+LN max-c ratio        : {tests['T2']['raw']['muon_max_c_ratio_no_ln_over_plus_ln']:.4f}x")
print(f"SGD-LN / SGD+LN max-c ratio          : {tests['T2']['raw']['sgd_max_c_ratio_no_ln_over_plus_ln']:.4f}x")
print("This figure supports discussion about observed norm balance only; it does not by itself establish a mechanism.")

## Loss trajectories

The next figure plots the full-batch mean-squared error over training for all four conditions. A log-scale y-axis is used because the late-stage losses differ by large multiplicative factors.

In [ ]:
plt.figure(figsize=(9, 4.8))
for name in meta["config_order"]:
    steps = np.array(results[name]["step"])
    loss_vals = np.array(results[name]["loss"])
    finite = np.isfinite(loss_vals) & (loss_vals > 0)
    plt.semilogy(steps[finite], loss_vals[finite], label=name, linewidth=2)
plt.xlabel("training step")
plt.ylabel("MSE loss (log scale)")
plt.title("H25a training-loss trajectories")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
ranking = sorted((results[name]["final_loss"], name) for name in meta["config_order"])
print("Interpretation of loss(t)")
print("-" * 27)
print("Final-loss ranking:")
for loss_value, name in ranking:
    print(f"  {name:>8}: {loss_value:.4f}")
print()
if results["Muon-LN"]["final_loss"] < results["Muon+LN"]["final_loss"]:
    print("Important honesty check: Muon-LN finishes below Muon+LN on final loss in this run.")
    print("So the evidence supports 'LN lowers Muon's observed c(t)' more strongly than 'LN improves Muon's final loss'.")
else:
    print("In this run, Muon+LN also wins on final loss among the Muon variants.")
print(f"T3 pass (Muon advantage within the +LN comparison): {tests['T3']['pass']}")
print(f"T4 pass (LN hurts SGD more than Muon): {tests['T4']['pass']}")

## Final per-layer Frobenius norms

The final bar chart shows the terminal layerwise weight norms. This is a descriptive diagnostic of how spread-out the weight magnitudes are at the end of training; it is not a substitute for the full `c(t)` trajectory.

In [ ]:
reference_norms = next(
    results[name]["norms_final"]
    for name in meta["config_order"]
    if results[name]["norms_final"]
)
layer_labels = list(reference_norms.keys())
x = np.arange(len(layer_labels))
width = 0.2

plt.figure(figsize=(11, 4.8))
for idx, name in enumerate(meta["config_order"]):
    if results[name]["norms_final"]:
        norm_values = [results[name]["norms_final"].get(layer, np.nan) for layer in layer_labels]
    else:
        norm_values = [np.nan] * len(layer_labels)
    plt.bar(x + (idx - 1.5) * width, norm_values, width=width, label=name)

plt.xticks(x, layer_labels, rotation=45, ha="right")
plt.ylabel(r"final $\|W_\ell\|_F$")
plt.title("Final per-layer Frobenius norms")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
print("Interpretation of final per-layer norms")
print("-" * 39)
for name in meta["config_order"]:
    if results[name]["norms_final"]:
        norm_values = list(results[name]["norms_final"].values())
        print(
            f"{name:>8}: min={min(norm_values):.4f}, max={max(norm_values):.4f}, spread={max(norm_values)/min(norm_values):.4f}"
        )
    else:
        print(f"{name:>8}: no final layer-norm profile because the run diverged before a finite endpoint.")
print("These final spreads should be read together with the trajectory plot, since transient imbalance can matter even when final norms look moderate.")


## Honest conclusion and limitations

This notebook should end with conclusions that match the actual returned metrics rather than a pre-written story.

In [ ]:
print("Conclusion")
print("=" * 10)
print(f"Test score: {tests['summary']['n_pass']}/{tests['summary']['score_out_of']}")
print()
print("Supported by this single run:")
print("- the pair now uses a truly shared initialization across all four conditions;")
print("- Muon+LN can be checked honestly against the c < 2 safe-zone criterion at every step;")
print("- removing LN hurts SGD much more than Muon according to the implemented T4 asymmetry metric;")
print("- within the +LN comparison, Muon beats SGD on final loss in this toy setup.")
print()
if results["Muon-LN"]["final_loss"] < results["Muon+LN"]["final_loss"]:
    print("Important caution:")
    print("- Muon-LN attains lower final loss than Muon+LN in this verified seed-42 run.")
    print("- Therefore it would be misleading to summarize the present evidence as 'LN+Muon is best overall'.")
else:
    print("In this run, Muon+LN does not lose to Muon-LN on final loss.")
print()
print("Not supported by this notebook:")
print("- any transformer-level claim;")
print("- any statement about robustness across seeds or hyperparameters;")
print("- any mechanistic claim involving gradients, activations, or gauge quantities not explicitly measured here.")
print()
print("Most useful next step if stronger evidence is wanted:")
print("- add a small multi-seed sweep while preserving the same structured script API.")

## Optional script-style text report

If desired, the following cell can reproduce the script's textual summary directly from the same `experiment` object already used above.

In [ ]:
module.print_report(experiment)